# Bài học 13 - Bộ nhớ đại lý với Đồ thị Kiến thức Cognee


## Thiết lập

Sổ tay này trình bày cách xây dựng một **trợ lý lập trình** thông minh với bộ nhớ lâu dài sử dụng đồ thị tri thức [**Cognee**](https://www.cognee.ai/) và **Khung tác vụ Microsoft Agent** (MAF).

Cognee biến văn bản không có cấu trúc thành đồ thị tri thức có cấu trúc, có thể truy vấn dựa trên các biểu diễn vector nhúng — cung cấp cho đại lý của bạn một bộ nhớ dài hạn phong phú và có nhận thức về mối quan hệ.

### Những gì bạn sẽ học
1. **Xây dựng Đồ thị Tri thức**: Biến các hồ sơ nhà phát triển và các thực hành tốt nhất thành kiến thức có cấu trúc, có thể truy vấn.
2. **Tích hợp Cognee với MAF**: Sử dụng các hàm `@tool` để cho phép đại lý MAF truy vấn đồ thị tri thức của Cognee.
3. **Hội thoại Có Nhận thức Phiên**: Duy trì ngữ cảnh qua nhiều câu hỏi trong cùng một phiên.
4. **Bộ nhớ Dài hạn**: Lưu giữ kiến thức quan trọng qua các phiên và truy xuất chúng trong các cuộc trò chuyện mới.

### Yêu cầu trước
- Python 3.9+
- Redis chạy cục bộ (`docker run -d -p 6379:6379 redis`) để quản lý phiên
- Khóa API LLM (ví dụ OpenAI) — đặt `LLM_API_KEY` trong `.env`
- `CACHING=true` trong `.env` (bắt buộc cho các phiên Cognee)
- Dự án Azure AI Foundry với mô hình chat đã triển khai
- `AZURE_AI_PROJECT_ENDPOINT` và `AZURE_AI_MODEL_DEPLOYMENT_NAME` trong `.env`
- Đã xác thực Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Các loại Bộ nhớ của Tác nhân

Sổ tay này khám phá cùng ba loại bộ nhớ từ sổ tay Bài học 13 chính, nhưng sử dụng Cognee làm backend bộ nhớ dài hạn:

| Loại bộ nhớ | Cơ chế | Thời gian tồn tại |
|---|---|---|
| **Làm việc** | `agent.create_session()` (MAF) | Một cuộc hội thoại |
| **Ngắn hạn** | Bộ nhớ đệm phiên Cognee (Redis) | Một phiên |
| **Dài hạn** | Đồ thị kiến thức Cognee + vector | Vĩnh viễn |

### Kiến trúc Bộ nhớ của Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Chuẩn Bị Không Gian Lưu Trữ Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Phần 1 — Xây dựng Cơ sở Kiến thức

Chúng tôi thu thập ba loại dữ liệu để tạo ra một cơ sở kiến thức toàn diện cho trợ lý lập trình của mình:

1. **Hồ sơ Nhà phát triển** — chuyên môn cá nhân và nền tảng kỹ thuật
2. **Các Thói quen Tốt nhất trong Python** — Zen của Python với các hướng dẫn thực tiễn
3. **Cuộc trò chuyện Lịch sử** — các phiên hỏi đáp trước đây giữa nhà phát triển và trợ lý AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Trực quan hóa Đồ thị Kiến thức

Cognee có thể hiển thị một hình ảnh HTML tương tác của các thực thể và mối quan hệ mà nó đã trích xuất.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Làm giàu trí nhớ với Memify

`memify()` phân tích đồ thị kiến thức và tạo ra các quy tắc thông minh — xác định các mẫu, các phương pháp tốt nhất và mối quan hệ giữa các khái niệm.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Phần 2 — Đại lý MAF với Công cụ Cognee

Bây giờ chúng ta tạo một đại lý MAF có thể truy vấn đồ thị tri thức của Cognee thông qua các hàm `@tool`. Điều này cho phép đại lý tận dụng toàn bộ sức mạnh của tìm kiếm ngữ nghĩa nhận biết đồ thị đồng thời duy trì ngữ cảnh hội thoại thông qua các phiên.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Bộ Nhớ Làm Việc với Các Phiên

`AgentSession` (được tạo qua `agent.create_session()`) cung cấp bộ nhớ làm việc trong một phiên. Đại lý có thể tham khảo lại các tin nhắn trước đó đồng thời truy vấn vào đồ thị tri thức lâu dài của Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Phiên Mới — Bộ Nhớ Dài Hạn Vẫn Tồn Tại

Bắt đầu một phiên mới sẽ xóa bộ nhớ làm việc, nhưng đồ thị kiến thức Cognee vẫn còn sẵn. Đại lý có thể truy xuất cùng một kiến thức dài hạn trong một cuộc trò chuyện hoàn toàn mới.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Tóm tắt

Trong sổ tay này, bạn đã xây dựng một trợ lý lập trình kết hợp **bộ nhớ làm việc của MAF** (`agent.create_session()`) với **đồ thị kiến thức dài hạn của Cognee**.

### Những gì bạn đã học được
1. **Xây dựng đồ thị kiến thức**: Cognee tiếp nhận văn bản phi cấu trúc và xây dựng một đồ thị + bộ nhớ vectơ.
2. **Làm phong phú đồ thị với memify**: Các sự kiện suy ra và mối quan hệ phong phú hơn trên đồ thị hiện có của bạn.
3. **Tích hợp MAF + Cognee**: Hàm `@tool` cho phép agent MAF truy vấn đồ thị của Cognee một cách tự nhiên.
4. **Bộ nhớ làm việc + bộ nhớ dài hạn**: `AgentSession` (thông qua `agent.create_session()`) cung cấp ngữ cảnh phiên trong khi Cognee cung cấp kiến thức bền vững.
5. **Tìm kiếm lọc với NodeSets**: Nhắm mục tiêu các tập hợp con cụ thể của đồ thị kiến thức (ví dụ chỉ nguyên tắc).

### Những điểm rút ra chính
- **Cognee** biến văn bản thô thành bộ nhớ có cấu trúc, nhận biết mối quan hệ — mạnh hơn nhiều so với kho dữ liệu vectơ phẳng.
- **Hàm `@tool`** tạo cầu nối giữa agent MAF và các hệ thống kiến thức bên ngoài một cách rõ ràng.
- **`AgentSession`** (thông qua `agent.create_session()`) giữ ngữ cảnh từng cuộc trò chuyện riêng biệt với kiến thức tồn tại lâu dài.
- Đồ thị kiến thức giống nhau phục vụ cho nhiều phiên và nhiều agent khác nhau.

### Ứng dụng thực tế
- **Trợ lý lập trình viên**: Đánh giá mã, phân tích sự cố, trợ lý kiến trúc
- **Trợ lý khách hàng trực tiếp**: Hỗ trợ qua tài liệu sản phẩm, câu hỏi thường gặp, và ghi chú CRM
- **Trợ lý chuyên gia nội bộ**: Trợ lý chính sách, pháp lý, hoặc bảo mật suy luận trên các hướng dẫn
- **Lớp dữ liệu thống nhất**: Kết hợp dữ liệu có cấu trúc và không có cấu trúc vào một đồ thị có thể truy vấn duy nhất

### Bước tiếp theo
- Thử nghiệm nhận thức thời gian trong Cognee
- Định nghĩa ontology OWL cho chất lượng đồ thị chuyên ngành
- Thêm vòng phản hồi người dùng để cải thiện truy xuất theo thời gian
- Mở rộng ra hệ thống đa agent chia sẻ cùng lớp bộ nhớ Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ ban đầu được coi là nguồn chính xác và uy tín nhất. Đối với thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
